# AMBER simulation setup for basic protein MD

Heavily based on the AMBER Tutorial 1 (Section 5), Tutorial 7, and the BioExcel biobb workflow.

The dashboard provides you with the molecule as `input.pdb`.
First, give it a nice name, specify the main simulation length, and adjust equilibration steps if necessary.

In [ ]:
name = "protein"  # give me a better name

nanoseconds = 0.05  # just for fun
nsteps = int(nanoseconds * 500000)  # assumes 2 fs timestep

nmin = 5000  # minimization cycles per stage
nheat = 25000  # heating steps (50 ps)
nnvt = 50000  # NVT equilibration steps (100 ps)
nnpt = 125000  # NPT equilibration steps (250 ps)
temp0 = 300.0  # target temperature (K)
salt_mM = 150.0  # salt concentration
buffer = 12.0  # solvation buffer (angstroms)

In [ ]:
import contextlib
import re
import statistics

import amber_wrapper as amb
import matplotlib.pyplot as plt
import nglview as nv

## Look at the input

Tune NGLView parameters if needed, inspect the input visually

In [ ]:
nv.show_file("input.pdb")

## Initial setup

### Prepare PDB

Run `pdb4amber` to clean the input: rename residues, handle disulfides, remove CONECT records.

In [ ]:
amb.pdb4amber(i="input.pdb", o=f"{name}_clean.pdb", reduce=True)

### Build topology and solvate with tleap

Generate force field, neutralize, solvate, and add salt.

In [ ]:
with open(f"{name}.leap.in", "w") as leap:
    leap.write(f"""\
source leaprc.protein.ff19SB
source leaprc.water.opc
protein = loadpdb {name}_clean.pdb
check protein
saveamberparm protein {name}_gas.prmtop {name}_gas.rst7
addions protein Na+ 0
addions protein Cl- 0
solvateoct protein OPCBOX {buffer}
quit
""")

# Salt calculation: run tleap once to get water count,
# then write a final script with salt and saveamberparm.
amb.tleap(f=f"{name}.leap.in")

# Parse leap.log for number of water residues
with open("leap.log") as log:
    content = log.read()
match = re.search(r"Added\s+(\d+)\s+residues", content)
if match:
    n_wat = int(match.group(1))
    n_ion_pairs = int(n_wat * salt_mM / 56000)
    print(f"Water molecules: {n_wat}, adding {n_ion_pairs} ion pairs")
else:
    n_ion_pairs = 0
    print("Could not detect water count; skipping salt buffer")

# Build final leap script in one f-string.
salt_cmd = f"addionsrand protein Na+ {n_ion_pairs} Cl- {n_ion_pairs}\n" if n_ion_pairs > 0 else ""
with open(f"{name}.leap.in", "w") as leap:
    leap.write(f"""\
source leaprc.protein.ff19SB
source leaprc.water.opc
protein = loadpdb {name}_clean.pdb
check protein
saveamberparm protein {name}_gas.prmtop {name}_gas.rst7
addions protein Na+ 0
addions protein Cl- 0
solvateoct protein OPCBOX {buffer}
{salt_cmd}saveamberparm protein {name}_solv.prmtop {name}_solv.rst7
quit
""")

amb.tleap(f=f"{name}.leap.in")

### Minimize — Stage 1 (restrained)

Relax solvent around the restrained protein.

In [ ]:
with open("min1.mdin", "w") as m:
    m.write(f"""&cntrl
  imin=1, ncyc={nmin // 2}, maxcyc={nmin}, ntmin=1,
  ntb=1, cut=10.0,
  ntr=1,
  restraint_wt=2.0,
  restraintmask=":@CA,C,N",
  ntpr=50,
/
""")

amb.pmemd(
    i="min1.mdin",
    o="min1.mdout",
    p=f"{name}_solv.prmtop",
    c=f"{name}_solv.rst7",
    r="min1.rst7",
    ref=f"{name}_solv.rst7",
    O=True,
)

### Minimize — Stage 2 (unrestrained)

Relax the entire system.

In [ ]:
with open("min2.mdin", "w") as m:
    m.write(f"""&cntrl
  imin=1, ncyc={nmin // 2}, maxcyc={nmin}, ntmin=1,
  ntb=1, cut=10.0,
  ntr=0,
  ntpr=50,
/
""")

amb.pmemd(i="min2.mdin", o="min2.mdout", p=f"{name}_solv.prmtop", c="min1.rst7", r="min2.rst7", O=True)

In [ ]:
# Convert restart to PDB for visualization
amb.ambpdb(p=f"{name}_solv.prmtop", c="min2.rst7", o="min2.pdb")
nv.show_file("min2.pdb")

## Equilibration

### Heat the system (NVT with restraints)

Ramp from 0 K to target temperature with weak restraints on the protein.

In [ ]:
with open("heat.mdin", "w") as h:
    h.write(f"""&cntrl
  imin=0, irest=0, ntx=1,
  ntb=1, cut=10.0,
  ntc=2, ntf=2,
  ntt=3, gamma_ln=1.0, ig=-1,
  tempi=0.0, temp0={temp0},
  ntr=1,
  restraint_wt=1.0,
  restraintmask=":@CA,C,N",
  nstlim={nheat}, dt=0.002,
  ntpr=100, ntwx=1000, ntwr=5000,
  ioutfm=1, iwrap=1, ntxo=2,
/
""")

amb.pmemd(
    i="heat.mdin",
    o="heat.mdout",
    p=f"{name}_solv.prmtop",
    c="min2.rst7",
    r="heat.rst7",
    ref=f"{name}_solv.rst7",
    O=True,
)

In [ ]:
# Extract temperature from mdout and plot
steps, temp = [], []
with open("heat.mdout") as f:
    in_results = False
    for line in f:
        if not in_results and line.startswith(" NSTEP ="):
            in_results = True
        if not in_results:
            continue
        if "A V E R A G E S" in line:
            break
        if "TEMP(K)" in line:
            parts = line.split()
            if len(parts) >= 9:
                with contextlib.suppress(ValueError):
                    nstep = int(parts[2])
                    t = float(parts[8])
                    if not steps or nstep > steps[-1]:
                        steps.append(nstep)
                        temp.append(t)

if temp:
    plt.figure(figsize=(10, 4))
    plt.plot(steps, temp)
    plt.title("Heating phase temperature")
    plt.xlabel("Step")
    plt.ylabel("Temperature (K)")
    plt.grid()
    plt.show()

### NVT equilibration (restrained)

Hold the box at constant volume while maintaining temperature.

In [ ]:
with open("nvt.mdin", "w") as nvt:
    nvt.write(f"""&cntrl
  imin=0, irest=1, ntx=5,
  ntb=1, cut=10.0,
  ntc=2, ntf=2,
  ntt=3, gamma_ln=1.0, ig=-1, temp0={temp0},
  ntr=1,
  restraint_wt=0.5,
  restraintmask=":@CA,C,N,O",
  nstlim={nnvt}, dt=0.002,
  ntpr=100, ntwx=1000, ntwr=5000,
  ioutfm=1, iwrap=1, ntxo=2,
/
""")

amb.pmemd(
    i="nvt.mdin", o="nvt.mdout", p=f"{name}_solv.prmtop", c="heat.rst7", r="nvt.rst7", ref=f"{name}_solv.rst7", O=True
)

In [ ]:
# Plot temperature from NVT equilibration
steps, temp = [], []
with open("nvt.mdout") as f:
    in_results = False
    for line in f:
        if not in_results and line.startswith(" NSTEP ="):
            in_results = True
        if not in_results:
            continue
        if "A V E R A G E S" in line:
            break
        if "TEMP(K)" in line:
            parts = line.split()
            if len(parts) >= 9:
                with contextlib.suppress(ValueError):
                    nstep = int(parts[2])
                    t = float(parts[8])
                    if not steps or nstep > steps[-1]:
                        steps.append(nstep)
                        temp.append(t)

if temp:
    plt.figure(figsize=(10, 4))
    plt.plot(steps, temp)
    plt.title("NVT equilibration temperature")
    plt.xlabel("Step")
    plt.ylabel("Temperature (K)")
    plt.grid()
    plt.show()
    mean_t = sum(temp) / len(temp)
    std_t = statistics.stdev(temp) if len(temp) > 1 else 0.0
    print(f"Mean T = {mean_t:.2f} K,  std = {std_t:.2f} K")

### NPT equilibration (unrestrained)

Allow the box volume to relax at constant pressure.

In [ ]:
with open("npt.mdin", "w") as npt:
    npt.write(f"""&cntrl
  imin=0, irest=1, ntx=5,
  ntb=2, cut=10.0,
  ntc=2, ntf=2,
  ntt=3, gamma_ln=1.0, ig=-1, temp0={temp0},
  ntp=1, barostat=2, pres0=1.0, taup=2.0,
  ntr=0,
  nstlim={nnpt}, dt=0.002,
  ntpr=100, ntwx=1000, ntwr=5000,
  ioutfm=1, iwrap=1, ntxo=2,
/
""")

amb.pmemd(i="npt.mdin", o="npt.mdout", p=f"{name}_solv.prmtop", c="nvt.rst7", r="npt.rst7", O=True)

In [ ]:
# Plot temperature and density from NPT (pressure is not computed with barostat=2)
data = {"Step": [], "Density": [], "Temp": []}
current = {}

with open("npt.mdout") as f:
    in_results = False
    for line in f:
        if not in_results and line.startswith(" NSTEP ="):
            in_results = True
        if not in_results:
            continue
        if "A V E R A G E S" in line:
            break
        parts = line.split()
        if line.startswith(" NSTEP =") and "TEMP(K)" in line:
            if current.get("step") is not None:
                data["Step"].append(current["step"])
                data["Temp"].append(current.get("temp"))
                data["Density"].append(current.get("density"))
            with contextlib.suppress(ValueError):
                current = {
                    "step": int(parts[2]),
                    "temp": float(parts[8]),
                }
        elif "DENSTY" in parts or "Density" in parts or "DENSITY" in parts:
            for i, p in enumerate(parts):
                if p in ("DENSTY", "Density", "DENSITY"):
                    with contextlib.suppress(ValueError, IndexError):
                        current["density"] = float(parts[i + 2])
                    break

if current.get("step") is not None:
    data["Step"].append(current["step"])
    data["Temp"].append(current.get("temp"))
    data["Density"].append(current.get("density"))

fig, axes = plt.subplots(2, 1, figsize=(10, 8))
for ax, key in zip(axes, ["Density", "Temp"]):
    vals = [v for v in data[key] if v is not None]
    if vals:
        ax.plot(data["Step"][:len(vals)], vals)
        ax.set_ylabel(key)
        ax.grid()
axes[-1].set_xlabel("Step")
plt.suptitle("NPT equilibration")
plt.tight_layout()
plt.show()

## Prepare production simulation

Run unrestrained production MD in the NPT ensemble.

In [ ]:
with open("prod.mdin", "w") as prod:
    prod.write(f"""&cntrl
  imin=0, irest=1, ntx=5,
  ntb=2, cut=10.0,
  ntc=2, ntf=2,
  ntt=3, gamma_ln=1.0, ig=-1, temp0={temp0},
  ntp=1, barostat=2, pres0=1.0,
  ntr=0,
  nstlim={nsteps}, dt=0.002,
  ntpr=5000, ntwx=5000, ntwr=5000,
  ioutfm=1, iwrap=1, ntxo=2,
/
""")

amb.pmemd(i="prod.mdin", o="prod.mdout", p=f"{name}_solv.prmtop", c="npt.rst7", r="prod.rst7", x="prod.nc", O=True)